# 01 · Environment setup and smoke test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/claverfred/real_time_swahili_english_interpreter/blob/main/notebooks/01_environment_setup.ipynb)

Prepares a fresh Colab runtime and checks every stage on the GPU:

1. clone (or update) the repository, mount Google Drive, Hugging Face login from Colab Secrets;
2. install the pinned extras **without replacing Colab's torch/numpy** (no restart needed);
3. download the evaluation data once (FLEURS Swahili test, FLORES-200; kept on Drive);
4. record the environment (GPU, CUDA, cuDNN, package versions, `pip freeze` lock file);
5. smoke test: config → MT → TTS → ASR on a real recording → full pipeline.

The first run downloads and converts the models (about 5-10 minutes). Later runs reuse them from Drive.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100/L40S are faster). Optional: add a Hugging Face token as the Colab Secret `HF_TOKEN` (key icon on the left); every default model is public, so it is not required.

Results, converted models and data are kept on Google Drive in `MyDrive/swahili_interpreter/`, so later sessions start faster. Nothing is uploaded anywhere else.

In [ ]:
# @title 1. Setup: clone/update the repository, mount Drive, install, fetch data (safe to re-run)
REPO_URL = "https://github.com/claverfred/real_time_swahili_english_interpreter.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
PROFILE = "colab"  # @param {type:"string"}

import os, subprocess, sys
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(WORKDIR, REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git"))
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %cd  %s"], capture_output=True, text=True).stdout)

from src.platform import colab
info = colab.bootstrap(REPO, use_drive=USE_DRIVE)
print("persistent root:", info["root"], "(Google Drive)" if info["on_drive"] else "")
for n in info["notes"]:
    print("NOTE:", n)
print("Hugging Face token:", "from Colab Secrets" if info["hf_login"] else "none (fine: all default models are public)")

r = subprocess.run([sys.executable, "scripts/setup_colab.py", "--profile", PROFILE])
stale = colab.restart_required()
if stale:
    print("Restart the runtime (Runtime > Restart session), then run this cell again:", stale)
assert r.returncode == 0, "setup reported a failure (see the table above)"

In [ ]:
# @title 2. GPU and package report
import json
rep = colab.environment_report()
print(rep["nvidia_smi"] or "nvidia-smi not found: no GPU attached (Runtime > Change runtime type)")
print(json.dumps({k: rep[k] for k in ("python", "packages", "torch_cudnn", "ctranslate2_cuda_devices",
                                      "ctranslate2_compute_types", "cuda_library_dirs")}, indent=2))
print("GPU memory:", colab.gpu_memory())

In [ ]:
# @title 3. Resolved configuration (device, precision, models) and diagnosis
!python -m src.cli config show --profile {PROFILE} --resolve | head -n 60
!python -m src.cli diagnose

In [ ]:
# @title 4. Dependency smoke test: the fast unit tests (no model downloads, ~1 minute)
!python -m pytest -q -m "not slow and not gpu" -p no:cacheprovider 2>&1 | tail -n 5

If every row above says `ok`, the other notebooks will run. A `warn` row explains itself (e.g. running on CPU). Re-running this notebook later only updates the repository and repeats the checks.

In [ ]:
# @title 5. Export the environment report and lock file
RUN_DIRS, EXTRA_FILES, EXPORT_NAME = [], [], "environment"
import time
zip_path = colab.results_dir() / f"{EXPORT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}.zip"
env_files = sorted((colab.results_dir().parent / "env").glob("*"))[-4:]
z = colab.zip_runs([p for p in RUN_DIRS if p], zip_path, extra_files=EXTRA_FILES + env_files)
print(z)
colab.download(zip_path)